<a href="https://colab.research.google.com/github/shehzadtm/KhuramFiles/blob/main/vertical_etf_money_flow_scanner_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Vertical ETF Money-Flow & Sector-Rotation Scanner

This Google Colab notebook ranks sector and thematic ETFs using:

- Relative strength versus SPY
- 1-, 3-, and 6-month momentum
- Chaikin Money Flow (CMF)
- On-Balance Volume (OBV) trend
- Volume expansion
- 20/50/200-day moving-average trend
- Median daily dollar-volume liquidity
- Multi-signal accumulation and exit warnings

> **Important:** These are price-and-volume proxies, not official ETF creation/redemption flows. Use the results as a screening and risk-management tool, not as proof of institutional buying or as personalized financial advice.


In [ ]:

# Install packages in Google Colab
!pip -q install yfinance plotly tabulate


## 1. Configuration

The notebook includes the 11 major U.S. sectors, important industry ETFs, and candidate funds for nuclear, cybersecurity, quantum computing, photonics/optics, and space.

For each specialized theme, it automatically keeps the candidate with the highest recent median dollar trading volume among symbols for which Yahoo Finance returns usable data.


In [ ]:

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import yfinance as yf
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, Markdown

BENCHMARK = "SPY"
PERIOD = "2y"
MIN_DOLLAR_VOLUME = 5_000_000

CORE_ETFS = {
    "XLK": "Technology",
    "XLF": "Financials",
    "XLV": "Health Care",
    "XLI": "Industrials",
    "XLE": "Energy",
    "XLY": "Consumer Discretionary",
    "XLP": "Consumer Staples",
    "XLC": "Communication Services",
    "XLU": "Utilities",
    "XLRE": "Real Estate",
    "XLB": "Materials",
    "SOXX": "Semiconductors",
    "XBI": "Biotechnology",
    "XAR": "Aerospace & Defense",
    "KRE": "Regional Banks",
    "XOP": "Oil & Gas Exploration",
    "XME": "Metals & Mining",
    "XHB": "Homebuilders",
    "BOTZ": "Robotics & Automation",
    "ICLN": "Clean Energy",
    "LIT": "Lithium & Batteries",
    "ARKK": "Disruptive Innovation",
}

SPECIALIZED_CANDIDATES = {
    "Nuclear / Uranium": ["URA", "NLR", "URNM", "NUKZ"],
    "Cybersecurity": ["CIBR", "HACK", "IHAK", "BUG"],
    "Quantum Computing": ["QTUM", "WQTM", "QNTM", "CQTM"],
    # Pure-play photonics ETFs are still an emerging category.
    # The scanner tests available symbols and can fall back to optics-heavy semiconductor exposure.
    "Photonics / Optics": ["EUV", "LYTE", "PHOT", "SOXX"],
    "Space": ["UFO", "ARKX", "ROKT"],
}

print("Universe configured.")

Universe configured.


## 2. Download market data

In [ ]:

def download_ohlcv(tickers, period="2y"):
    tickers = sorted(set(tickers))
    raw = yf.download(
        tickers=tickers,
        period=period,
        interval="1d",
        auto_adjust=True,
        group_by="ticker",
        threads=True,
        progress=False,
    )

    result = {}
    for ticker in tickers:
        try:
            frame = raw.copy() if len(tickers) == 1 else raw[ticker].copy()
            required = ["Open", "High", "Low", "Close", "Volume"]
            if not all(c in frame.columns for c in required):
                continue
            frame = frame[required].dropna(subset=["Close"])
            frame["Volume"] = frame["Volume"].fillna(0)
            frame = frame[(frame["Close"] > 0) & (frame["Volume"] >= 0)]
            if len(frame) >= 30:
                result[ticker] = frame
        except Exception:
            pass
    return result

all_candidates = list(CORE_ETFS)
for group in SPECIALIZED_CANDIDATES.values():
    all_candidates.extend(group)
all_candidates.append(BENCHMARK)

data = download_ohlcv(all_candidates, PERIOD)

print(f"Downloaded usable data for {len(data)} symbols.")
if BENCHMARK not in data:
    raise RuntimeError("SPY benchmark data could not be downloaded. Re-run the cell.")

ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: PHOT"}}}
ERROR:yfinance:
2 Failed downloads:
ERROR:yfinance:['PHOT']: YFPricesMissingError('possibly delisted; no price data found  (period=2y) (Yahoo error = "No data found, symbol may be delisted")')
ERROR:yfinance:['LYTE']: YFPricesMissingError('possibly delisted; no price data found  (period=2y)')


Downloaded usable data for 39 symbols.


## 3. Select the most liquid ETF in each specialized theme

In [ ]:

def median_dollar_volume(frame, window=20):
    return float((frame["Close"] * frame["Volume"]).tail(window).median())

selected_specialized = {}
liquidity_rows = []

for vertical, candidates in SPECIALIZED_CANDIDATES.items():
    available = []
    for ticker in candidates:
        value = median_dollar_volume(data[ticker]) if ticker in data else 0.0
        liquidity_rows.append({
            "Vertical": vertical,
            "Ticker": ticker,
            "Median $ Volume (20D)": value,
            "Available": ticker in data,
        })
        if ticker in data and value > 0:
            available.append((ticker, value))

    if available:
        selected_specialized[max(available, key=lambda x: x[1])[0]] = vertical

liquidity_table = pd.DataFrame(liquidity_rows)
liquidity_table["Selected"] = liquidity_table.apply(
    lambda r: selected_specialized.get(r["Ticker"]) == r["Vertical"], axis=1
)
liquidity_table["Median $ Volume (20D, $M)"] = (
    liquidity_table["Median $ Volume (20D)"] / 1_000_000
)

display(
    liquidity_table.sort_values(
        ["Vertical", "Median $ Volume (20D)"], ascending=[True, False]
    )[["Vertical", "Ticker", "Median $ Volume (20D, $M)", "Available", "Selected"]]
)

print("\nSelected specialized ETFs:")
for ticker, vertical in selected_specialized.items():
    print(f"{vertical:24s} -> {ticker}")

,Vertical,Ticker,"Median $ Volume (20D, $M)",Available,Selected
4,Cybersecurity,CIBR,128.056323,True,True
7,Cybersecurity,BUG,52.536006,True,False
5,Cybersecurity,HACK,19.957967,True,False
6,Cybersecurity,IHAK,9.624715,True,False
0,Nuclear / Uranium,URA,115.960133,True,True
1,Nuclear / Uranium,NLR,47.872119,True,False
2,Nuclear / Uranium,URNM,19.620260,True,False
3,Nuclear / Uranium,NUKZ,5.722889,True,False
15,Photonics / Optics,SOXX,5617.240754,True,True
12,Photonics / Optics,EUV,19.908281,True,False



Selected specialized ETFs:
Nuclear / Uranium        -> URA
Cybersecurity            -> CIBR
Quantum Computing        -> QTUM
Photonics / Optics       -> SOXX
Space                    -> UFO


## 4. Technical indicators and scoring model

In [ ]:

def safe_return(series, lookback):
    if len(series) <= lookback:
        return np.nan
    start, end = float(series.iloc[-lookback - 1]), float(series.iloc[-1])
    return end / start - 1 if start > 0 else np.nan

def rsi(close, window=14):
    delta = close.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.ewm(alpha=1/window, adjust=False, min_periods=window).mean()
    avg_loss = loss.ewm(alpha=1/window, adjust=False, min_periods=window).mean()
    rs = avg_gain / avg_loss.replace(0, np.nan)
    value = 100 - (100 / (1 + rs))
    return value.where(avg_loss != 0, 100)

def cmf(frame, window=20):
    spread = (frame["High"] - frame["Low"]).replace(0, np.nan)
    multiplier = (
        (frame["Close"] - frame["Low"]) -
        (frame["High"] - frame["Close"])
    ) / spread
    mf_volume = multiplier.fillna(0) * frame["Volume"]
    return mf_volume.rolling(window).sum() / frame["Volume"].rolling(window).sum().replace(0, np.nan)

def obv(frame):
    direction = np.sign(frame["Close"].diff()).fillna(0)
    return (direction * frame["Volume"]).cumsum()

def normalized_slope(series, window=20):
    values = series.dropna().tail(window).to_numpy(dtype=float)
    if len(values) < 10:
        return np.nan
    denominator = np.mean(np.abs(values))
    if denominator == 0:
        return 0.0
    return float(np.polyfit(np.arange(len(values)), values, 1)[0] / denominator)

def accumulation_balance(frame, window=20):
    recent = frame.tail(window).copy()
    recent["Return"] = recent["Close"].pct_change()
    recent["AvgVolume"] = frame["Volume"].rolling(50).mean().tail(window)
    high_volume = recent["Volume"] > recent["AvgVolume"]
    accumulation = ((recent["Return"] > 0) & high_volume).sum()
    distribution = ((recent["Return"] < 0) & high_volume).sum()
    return float((accumulation - distribution) / window)

def pct_score(series, higher_is_better=True):
    score = series.rank(pct=True, method="average")
    if not higher_is_better:
        score = 1 - score
    return score.fillna(0.5) * 100

def analyze_ticker(ticker, vertical, frame, benchmark):
    close = frame["Close"]
    volume = frame["Volume"]
    benchmark_close = benchmark["Close"].reindex(frame.index).ffill()

    sma20 = close.rolling(20).mean()
    sma50 = close.rolling(50).mean()
    sma200 = close.rolling(200).mean()
    cmf_series = cmf(frame)
    obv_series = obv(frame)

    latest = float(close.iloc[-1])
    s20 = float(sma20.iloc[-1]) if pd.notna(sma20.iloc[-1]) else np.nan
    s50 = float(sma50.iloc[-1]) if pd.notna(sma50.iloc[-1]) else np.nan
    s200 = float(sma200.iloc[-1]) if pd.notna(sma200.iloc[-1]) else np.nan

    r1, r3, r6 = [safe_return(close, n) for n in (21, 63, 126)]
    b1, b3, b6 = [safe_return(benchmark_close.dropna(), n) for n in (21, 63, 126)]

    avg20 = float(volume.tail(20).mean())
    avg60 = float(volume.tail(60).mean())
    high63 = float(close.tail(63).max())

    return {
        "Ticker": ticker,
        "Vertical": vertical,
        "Close": latest,
        "Return1M": r1,
        "Return3M": r3,
        "Return6M": r6,
        "RS1MvsSPY": r1 - b1,
        "RS3MvsSPY": r3 - b3,
        "RS6MvsSPY": r6 - b6,
        "RSI14": float(rsi(close).iloc[-1]),
        "CMF20": float(cmf_series.iloc[-1]),
        "OBVSlope20": normalized_slope(obv_series, 20),
        "AccumulationBalance20": accumulation_balance(frame, 20),
        "VolumeExpansion20vs60": avg20 / avg60 - 1 if avg60 else np.nan,
        "MedianDollarVolume20D": median_dollar_volume(frame),
        "DistanceFrom63DHigh": latest / high63 - 1 if high63 else np.nan,
        "AboveSMA20": latest > s20 if np.isfinite(s20) else False,
        "AboveSMA50": latest > s50 if np.isfinite(s50) else False,
        "AboveSMA200": latest > s200 if np.isfinite(s200) else False,
        "BullishMAAlignment": (
            np.isfinite(s20) and np.isfinite(s50) and np.isfinite(s200)
            and s20 > s50 > s200
        ),
        "BearishMAAlignment": (
            np.isfinite(s20) and np.isfinite(s50)
            and s20 < s50 and latest < s50
        ),
    }

universe = dict(CORE_ETFS)
universe.update(selected_specialized)

records = []
for ticker, vertical in universe.items():
    if ticker in data:
        records.append(analyze_ticker(ticker, vertical, data[ticker], data[BENCHMARK]))

results = pd.DataFrame(records)

results["MomentumScore"] = (
    0.20 * pct_score(results["Return1M"]) +
    0.35 * pct_score(results["Return3M"]) +
    0.45 * pct_score(results["Return6M"])
)
results["RelativeStrengthScore"] = (
    0.20 * pct_score(results["RS1MvsSPY"]) +
    0.35 * pct_score(results["RS3MvsSPY"]) +
    0.45 * pct_score(results["RS6MvsSPY"])
)
results["MoneyFlowScore"] = (
    0.40 * pct_score(results["CMF20"]) +
    0.30 * pct_score(results["OBVSlope20"]) +
    0.20 * pct_score(results["AccumulationBalance20"]) +
    0.10 * pct_score(results["VolumeExpansion20vs60"])
)
trend_points = (
    results["AboveSMA20"].astype(int) +
    results["AboveSMA50"].astype(int) +
    results["AboveSMA200"].astype(int) +
    results["BullishMAAlignment"].astype(int)
)
results["TrendScore"] = trend_points / 4 * 100
results["LiquidityScore"] = pct_score(
    np.log10(results["MedianDollarVolume20D"].clip(lower=1))
)
results["RSIQuality"] = np.select(
    [
        results["RSI14"].between(50, 70),
        results["RSI14"].between(40, 50, inclusive="left"),
        results["RSI14"].between(70, 80, inclusive="right"),
        results["RSI14"] < 40,
        results["RSI14"] > 80,
    ],
    [100, 65, 55, 25, 20],
    default=50,
)
results["CompositeScore"] = (
    0.25 * results["RelativeStrengthScore"] +
    0.20 * results["MomentumScore"] +
    0.25 * results["MoneyFlowScore"] +
    0.20 * results["TrendScore"] +
    0.05 * results["LiquidityScore"] +
    0.05 * results["RSIQuality"]
)
results["LiquidEnough"] = results["MedianDollarVolume20D"] >= MIN_DOLLAR_VOLUME

strong = (
    (results["CompositeScore"] >= 70) &
    (results["CMF20"] > 0.05) &
    (results["RS3MvsSPY"] > 0) &
    results["AboveSMA50"] &
    results["LiquidEnough"]
)
accumulating = (
    (results["CompositeScore"] >= 58) &
    (results["CMF20"] > 0) &
    (results["RS3MvsSPY"] > 0) &
    results["AboveSMA50"] &
    results["LiquidEnough"]
)
distribution = (
    (results["CMF20"] < -0.05) |
    (results["OBVSlope20"] < 0) |
    (results["RS3MvsSPY"] < 0) |
    results["BearishMAAlignment"]
)

results["FlowRegime"] = np.select(
    [strong, accumulating, distribution],
    ["STRONG ACCUMULATION", "ACCUMULATION", "DISTRIBUTION"],
    default="NEUTRAL / WATCH",
)
results["ExitWarning"] = (
    ((~results["AboveSMA50"]) & (results["CMF20"] < 0) & (results["RS1MvsSPY"] < 0))
    | (results["BearishMAAlignment"] & (results["OBVSlope20"] < 0))
    | ((results["DistanceFrom63DHigh"] < -0.12) & (results["RS3MvsSPY"] < 0))
)

results = results.sort_values("CompositeScore", ascending=False).reset_index(drop=True)
print("Scoring complete.")

Scoring complete.


## 5. Ranked dashboard

In [ ]:

display_cols = [
    "Ticker", "Vertical", "FlowRegime", "CompositeScore",
    "MoneyFlowScore", "RelativeStrengthScore", "TrendScore",
    "Return3M", "RS3MvsSPY", "CMF20", "RSI14",
    "MedianDollarVolume20D", "ExitWarning"
]

styled = (
    results[display_cols]
    .style
    .format({
        "CompositeScore": "{:.1f}",
        "MoneyFlowScore": "{:.1f}",
        "RelativeStrengthScore": "{:.1f}",
        "TrendScore": "{:.1f}",
        "Return3M": "{:+.1%}",
        "RS3MvsSPY": "{:+.1%}",
        "CMF20": "{:+.2f}",
        "RSI14": "{:.1f}",
        "MedianDollarVolume20D": "${:,.0f}",
    })
    .background_gradient(subset=["CompositeScore", "MoneyFlowScore", "RelativeStrengthScore"])
)
display(styled)

,Ticker,Vertical,FlowRegime,CompositeScore,MoneyFlowScore,RelativeStrengthScore,TrendScore,Return3M,RS3MvsSPY,CMF20,RSI14,MedianDollarVolume20D,ExitWarning
0,XLE,Energy,DISTRIBUTION,85.6,86.9,76.2,100.0,+0.6%,-3.7%,+0.03,63.6,"$1,780,151,128",False
1,XOP,Oil & Gas Exploration,DISTRIBUTION,85.0,87.7,79.0,100.0,+0.0%,-4.2%,+0.10,64.5,"$520,408,412",False
2,XLF,Financials,STRONG ACCUMULATION,84.8,83.5,75.8,100.0,+9.6%,+5.4%,+0.10,62.3,"$1,832,875,468",False
3,KRE,Regional Banks,STRONG ACCUMULATION,82.1,78.1,75.8,100.0,+9.5%,+5.3%,+0.11,56.2,"$904,684,399",False
4,CIBR,Cybersecurity,DISTRIBUTION,80.4,46.9,93.7,100.0,+36.3%,+32.0%,-0.03,58.0,"$128,056,323",False
5,XBI,Biotechnology,ACCUMULATION,74.4,70.4,76.7,75.0,+12.0%,+7.8%,+0.02,43.8,"$1,255,059,673",False
6,XLV,Health Care,DISTRIBUTION,73.2,41.9,74.4,100.0,+11.8%,+7.6%,-0.10,55.1,"$1,540,259,150",False
7,XLP,Consumer Staples,DISTRIBUTION,72.7,64.2,62.1,100.0,+1.6%,-2.6%,-0.19,51.6,"$935,081,047",False
8,XLRE,Real Estate,DISTRIBUTION,68.8,39.6,71.0,100.0,+2.4%,-1.8%,-0.20,50.8,"$192,566,848",False
9,XLI,Industrials,DISTRIBUTION,67.7,66.5,64.4,75.0,+3.3%,-0.9%,-0.03,49.8,"$1,169,396,280",False


In [ ]:

fig = px.bar(
    results.sort_values("CompositeScore"),
    x="CompositeScore",
    y="Ticker",
    orientation="h",
    color="FlowRegime",
    hover_data=["Vertical", "CMF20", "RS3MvsSPY", "RSI14", "ExitWarning"],
    title="Vertical ETF Rotation Ranking",
)
fig.add_vline(x=70, line_dash="dash", annotation_text="Strong threshold")
fig.add_vline(x=58, line_dash="dot", annotation_text="Accumulation threshold")
fig.update_layout(height=max(650, 25 * len(results)), xaxis_range=[0, 100])
fig.show()


## 6. Money-flow vs. relative-strength map

In [ ]:

plot_df = results.copy()
plot_df["DollarVolumeMillions"] = plot_df["MedianDollarVolume20D"] / 1_000_000

fig = px.scatter(
    plot_df,
    x="RS3MvsSPY",
    y="CMF20",
    size="DollarVolumeMillions",
    color="CompositeScore",
    text="Ticker",
    hover_name="Vertical",
    hover_data=["FlowRegime", "RSI14", "ExitWarning"],
    title="Money Flow vs. 3-Month Relative Strength",
    labels={
        "RS3MvsSPY": "3-Month Return Minus SPY",
        "CMF20": "20-Day Chaikin Money Flow",
        "DollarVolumeMillions": "Median Daily $ Volume ($M)",
    },
)
fig.add_hline(y=0, line_dash="dash")
fig.add_vline(x=0, line_dash="dash")
fig.update_traces(textposition="top center")
fig.update_layout(height=700)
fig.show()


**How to read the chart**

- **Upper-right:** positive money flow and outperformance — strongest area.
- **Lower-right:** still outperforming, but current distribution may be starting.
- **Upper-left:** accumulation may be beginning before relative strength confirms.
- **Lower-left:** weak relative strength and negative money flow — generally the least attractive area.


## 7. Technical heatmap

In [ ]:

heat = results.set_index("Ticker")[
    ["MomentumScore", "RelativeStrengthScore", "MoneyFlowScore", "TrendScore", "CompositeScore"]
]

fig = px.imshow(
    heat,
    text_auto=".0f",
    aspect="auto",
    color_continuous_scale="RdYlGn",
    zmin=0,
    zmax=100,
    title="Cross-Sectional Technical Score Heatmap",
)
fig.update_layout(height=max(650, 25 * len(heat)))
fig.show()

## 8. Inspect any ETF visually

In [ ]:

# Change this ticker to inspect another ETF from the ranked table.
SELECTED_TICKER = results.iloc[0]["Ticker"]
print("Selected:", SELECTED_TICKER, "-", results.iloc[0]["Vertical"])

Selected: XLE - Energy


In [ ]:

def plot_etf_dashboard(ticker):
    if ticker not in data:
        raise ValueError(f"No data found for {ticker}")

    frame = data[ticker].copy()
    frame["SMA20"] = frame["Close"].rolling(20).mean()
    frame["SMA50"] = frame["Close"].rolling(50).mean()
    frame["SMA200"] = frame["Close"].rolling(200).mean()
    frame["CMF20"] = cmf(frame)
    frame["OBV"] = obv(frame)

    benchmark_close = data[BENCHMARK]["Close"].reindex(frame.index).ffill()
    frame["RelativeStrength"] = frame["Close"] / benchmark_close
    frame["RS_SMA50"] = frame["RelativeStrength"].rolling(50).mean()

    fig = make_subplots(
        rows=4, cols=1, shared_xaxes=True,
        vertical_spacing=0.035,
        row_heights=[0.48, 0.17, 0.17, 0.18],
        subplot_titles=(
            f"{ticker} Price and Moving Averages",
            "Volume",
            "Chaikin Money Flow",
            f"Relative Strength Ratio vs. {BENCHMARK}",
        ),
    )

    fig.add_trace(go.Candlestick(
        x=frame.index, open=frame["Open"], high=frame["High"],
        low=frame["Low"], close=frame["Close"], name=ticker
    ), row=1, col=1)
    for col in ["SMA20", "SMA50", "SMA200"]:
        fig.add_trace(go.Scatter(
            x=frame.index, y=frame[col], name=col, mode="lines"
        ), row=1, col=1)

    fig.add_trace(go.Bar(
        x=frame.index, y=frame["Volume"], name="Volume"
    ), row=2, col=1)

    fig.add_trace(go.Scatter(
        x=frame.index, y=frame["CMF20"], name="CMF20", mode="lines"
    ), row=3, col=1)
    fig.add_hline(y=0, line_dash="dash", row=3, col=1)
    fig.add_hline(y=0.05, line_dash="dot", row=3, col=1)
    fig.add_hline(y=-0.05, line_dash="dot", row=3, col=1)

    fig.add_trace(go.Scatter(
        x=frame.index, y=frame["RelativeStrength"],
        name="RS Ratio", mode="lines"
    ), row=4, col=1)
    fig.add_trace(go.Scatter(
        x=frame.index, y=frame["RS_SMA50"],
        name="RS 50D Average", mode="lines"
    ), row=4, col=1)

    fig.update_layout(
        height=1100,
        title=f"{ticker} Technical and Money-Flow Dashboard",
        xaxis_rangeslider_visible=False,
        hovermode="x unified",
    )
    fig.show()

plot_etf_dashboard(SELECTED_TICKER)

## 9. Current leaders and exit warnings

In [ ]:

leaders = results[
    results["FlowRegime"].isin(["STRONG ACCUMULATION", "ACCUMULATION"])
][["Ticker", "Vertical", "FlowRegime", "CompositeScore", "CMF20", "RS3MvsSPY"]]

warnings_df = results[results["ExitWarning"]][
    ["Ticker", "Vertical", "FlowRegime", "CompositeScore", "CMF20", "RS1MvsSPY", "AboveSMA50"]
]

display(Markdown("### Accumulation leaders"))
display(leaders.style.format({
    "CompositeScore": "{:.1f}", "CMF20": "{:+.2f}", "RS3MvsSPY": "{:+.1%}"
}))

display(Markdown("### Multi-signal exit warnings"))
display(warnings_df.style.format({
    "CompositeScore": "{:.1f}", "CMF20": "{:+.2f}", "RS1MvsSPY": "{:+.1%}"
}))

### Accumulation leaders

,Ticker,Vertical,FlowRegime,CompositeScore,CMF20,RS3MvsSPY
2,XLF,Financials,STRONG ACCUMULATION,84.8,+0.10,+5.4%
3,KRE,Regional Banks,STRONG ACCUMULATION,82.1,+0.11,+5.3%
5,XBI,Biotechnology,ACCUMULATION,74.4,+0.02,+7.8%


### Multi-signal exit warnings

,Ticker,Vertical,FlowRegime,CompositeScore,CMF20,RS1MvsSPY,AboveSMA50
10,XLK,Technology,DISTRIBUTION,66.2,+0.07,-5.7%,False
11,SOXX,Photonics / Optics,DISTRIBUTION,53.8,-0.12,-16.0%,False
13,XLB,Materials,DISTRIBUTION,47.1,-0.09,-1.3%,False
14,QTUM,Quantum Computing,DISTRIBUTION,46.2,-0.19,-12.1%,False
15,XLU,Utilities,DISTRIBUTION,39.0,-0.34,-1.1%,False
16,XAR,Aerospace & Defense,DISTRIBUTION,38.2,-0.09,-7.3%,False
17,XLC,Communication Services,DISTRIBUTION,31.7,-0.10,-1.5%,False
18,XHB,Homebuilders,DISTRIBUTION,30.6,-0.26,-8.1%,False
19,XME,Metals & Mining,DISTRIBUTION,28.1,+0.06,-3.7%,False
20,LIT,Lithium & Batteries,DISTRIBUTION,26.4,+0.02,-11.4%,False


## 10. Export results

In [ ]:

results.to_csv("vertical_etf_rankings.csv", index=False)
liquidity_table.to_csv("specialized_etf_liquidity.csv", index=False)

print("Saved:")
print("- vertical_etf_rankings.csv")
print("- specialized_etf_liquidity.csv")

# In Colab, uncomment these lines to download:
# from google.colab import files
# files.download("vertical_etf_rankings.csv")
# files.download("specialized_etf_liquidity.csv")

Saved:
- vertical_etf_rankings.csv
- specialized_etf_liquidity.csv



## Suggested operating discipline

A conservative process could be:

1. Run the notebook once per week after the U.S. market closes.
2. Focus on liquid ETFs in the upper-right quadrant of the flow/relative-strength chart.
3. Require the accumulation regime to persist for two weekly observations.
4. Position-size modestly because thematic ETFs can be concentrated and volatile.
5. Review or reduce a position when the exit warning persists, especially below the 50-day moving average.
6. Avoid treating a single indicator or one-day volume spike as decisive.

Official ETF fund flows require shares-outstanding or creation/redemption data. This notebook deliberately labels price-volume indicators as proxies.
